# Candidate Recovery Metric — Sleep Duration and Efficiency

Sleep is the wearable signal patients themselves most associate with recovery, and it is recorded
passively overnight rather than requiring the patient to move. Against that, **it is the signal
most easily disrupted by things that are not recovery** — a hospital ward, analgesia, a device
removed at night — so its face validity as a recovery measure is weaker than its coverage
suggests. Efficiency is carried alongside duration because the two can disagree, and disagreement
would mean two candidate metrics rather than one.

**Every reading here is within patient.** A per-day median over whoever recorded a night is
cross-sectional: it moves when the set of people wearing a device moves, so a peri-operative dip
in one is as consistent with the best sleepers taking the device off as with anyone sleeping
worse. This notebook therefore reads every night against that patient's own normal, using the four
baselines in `baselines.py` — a frozen pre-operative lookback, a trailing flat window, a trailing
exponential window, and a short window read against a long one. The cross-sectional curve appears
once, dotted, as the context the baselines are there to replace.

**What this does not settle is which baseline to use.** The four disagree structurally rather than
noisily, and what each costs in patients and person-days is Part 2. Their parameters were swept
against a cancer-diagnosis index in another project and are inherited rather than re-derived, which
`baselines.py` states in its own docstring.

**Governance.** The person-day extract is participant-level and stays in the kernel; every printed
number and every figure comes through `srp_common.reportable_days`, which drops any day fewer than
`SUPPRESS_THRESHOLD` patients contributed to. Unsuppressed figures stay git-ignored while
`*_suppressed.png` is committable, and the threshold must be confirmed against current All of Us
policy before anything is published.

## Shared Imports and Methods

`srp_common` resolves the pinned CDR, creates this notebook's git-ignored figure and cache
directories, and builds the cohort plus one index date per patient. The sleep CTE and every
baseline parameter the notebook runs under are settled here, so nothing below can fail on setup.
There is no analysis in this section and nothing here answers a question.

#### Imports, the Sleep CTE, and the Parameters Every Baseline Resolves To

`is_main_sleep` is a STRING in this CDR rather than a BOOL, so it is cast before comparison —
assume nothing about a column's type, look it up with `srp_common.column_type`. **The grid reaches
much further back than the span any figure is drawn over**, because a trailing baseline on the
first reported day needs its own history behind it and the static window is anchored to the surgery
rather than to that day. `baselines.grid_first_day` takes both reaches, so a parameter change moves
the extract rather than silently emptying a baseline.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from the kernel's cwd to the repo root, so this notebook runs from its task
# directory or from the root without the path being hard-coded in either.
_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import baselines as B
import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD, SURFACE

cfg = srp_common.init("fitbit_sleep")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq
reportable = srp_common.reportable_days

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used until then
# because it is the only rule no post-operative event can influence, and every result below is
# stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS
BASELINE_STATISTIC = B.BASELINE_CENTRE   # "median": both measures are skewed and bounded below
REPORT_FIRST, REPORT_LAST = -30, 30      # the span every figure is drawn over
ZOOM = (-10, 10)                         # the admission itself, at readable width
# How far back the GRID must reach for all four baselines to exist on REPORT_FIRST. Not simply
# REPORT_FIRST minus a window length: the static window is anchored to the surgery and the
# trailing windows to the day they score, so grid_first_day takes both reaches.
GRID_FIRST = B.grid_first_day(REPORT_FIRST)

# Declared before the query runs, so a renamed column fails here with a near-miss suggestion
# rather than as a BigQuery 400 partway through an extract.
cfg.require_columns({
    "sleep_daily_summary": ["person_id", "sleep_date", "is_main_sleep",
                            "minute_in_bed", "minute_asleep"],
})

METRIC_CTE = f"""
    sleep_nights AS (
        -- ONE ROW PER PERSON-DAY, NOT PER SLEEP EPISODE. sleep_daily_summary carries a row per
        -- episode, so a night split across two main-sleep records would pivot into a duplicate
        -- index entry and raise. Summing both sides of the ratio before dividing also gives the
        -- night's true efficiency, which averaging two per-episode ratios would not.
        SELECT
            s.person_id,
            DATE_DIFF(s.sleep_date, i.index_date, DAY) AS rel_day,
            SUM(s.minute_in_bed) AS minute_in_bed,
            SUM(s.minute_asleep) AS minute_asleep
        FROM `{CDR}`.sleep_daily_summary AS s
        JOIN index_surgery AS i ON i.person_id = s.person_id
        -- is_main_sleep is a STRING ('true'/'false') in this CDR, not a BOOL, so a bare
        -- `= true` is a type error; casting first covers a 1/0 encoding too.
        WHERE LOWER(CAST(s.is_main_sleep AS STRING)) IN ('true', '1')
          AND s.minute_asleep IS NOT NULL
          AND DATE_DIFF(s.sleep_date, i.index_date, DAY)
              BETWEEN {GRID_FIRST} AND {REPORT_LAST}
        GROUP BY s.person_id, rel_day
    )
"""
CTES = cfg.cohort_ctes(index_rule=INDEX_RULE, extra=METRIC_CTE)

print(f"index rule           : {INDEX_RULE}")
print(f"baseline statistic   : {BASELINE_STATISTIC}")
print("windows              : "
      + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))
print(f"reported span        : rel_day {REPORT_FIRST:+d} .. {REPORT_LAST:+d}")
print(f"grid spans           : rel_day {GRID_FIRST:+d} .. {REPORT_LAST:+d}"
      f"   ({REPORT_LAST - GRID_FIRST + 1} days)")
print()
print("the four baselines, at the values inherited from the sweeps")
print(f"  static      : {B.STATIC_LOOKBACK_DAYS}d window ending day "
      f"{-B.STATIC_DAYS_PRIOR:+d}, i.e. rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. "
      f"{-B.STATIC_DAYS_PRIOR:+d}, needing {B.STATIC_MIN_ELIGIBLE_DAYS} eligible days")
print(f"  rolling     : {B.ROLLING_WINDOW_DAYS}d trailing window, gap "
      f"{B.ROLLING_GAP_DAYS}d, needing {B.ROLLING_MIN_DAYS} of its "
      f"{B.ROLLING_WINDOW_DAYS} days ({1 - B.ROLLING_MAX_MISSING:.0%} of them)")
print(f"  ewma        : halflife {B.EWMA_HALFLIFE_DAYS}d, capacity "
      f"{B.ewma_full_effective_days():.1f} effective days, needing "
      f"{B.EWMA_MIN_EFFECTIVE_DAYS:.1f} of them")
print(f"  comparative : {B.SHORT_SPAN_DAYS}d short vs {B.LONG_SPAN_DAYS}d long, "
      f"{B.SPAN_COUPLING}, in units of the long window's IQR")
print()
print(f"small-cell threshold : {SUPPRESS_THRESHOLD} patients per reported day")

## Part 1 — The Person-Day Grids

Everything below runs on two objects: a grid of minutes asleep and a grid of sleep efficiency, each
with one row per `rel_day` and one column per patient, holding that patient's value on the nights
the eligibility rule admits and nothing anywhere else. **A grid row is a calendar day, not an
observed night** — that is the property the whole module depends on, because a trailing 30-day
window has to span the 30 days it claims, and if rows were observed nights a patient who wore the
device twice in a month would get a "30-day" window reaching back most of a year.

The two grids are not the same shape and that is deliberate. Efficiency needs minutes in bed as
well as minutes asleep, so a patient can clear a month of recorded nights while carrying only a
handful of usable efficiency values, and a baseline built from a handful is not a baseline. Keeping
them separate is what lets the two measures be compared honestly instead of being forced onto one
denominator.

This is the one Part that holds participant-level values in the kernel. That is allowed — the
analysis runs inside the Verily workspace and the rows never leave it — but it is the reason every
number printed from here on comes through `reportable()`.

#### Coverage, the Eligibility Rule, and What Each Bound Removes

Coverage comes first because no amount of signal rescues a metric the cohort does not carry, and
the paired count is read off the grids rather than re-queried, so the population reported here is
exactly the one the baselines run over. **Each eligibility bound is a judgement about what the
device was doing, not about what the patient was doing**, so every one of them is stated with the
person-days it removes rather than buried in the query. The contributor curve is the denominator
behind every median below and the quantity the small-cell threshold is applied to.

In [ ]:
# The eligibility rules, one per measure. Duration only needs the night itself to be plausible;
# efficiency additionally needs a usable time in bed, so its grid is a subset of duration's.
MIN_MINUTES_ASLEEP = 60     # a main sleep with under an hour of sleep is a device artifact
MAX_MINUTES_ASLEEP = 960    # over sixteen hours asleep is a device left in the bed, not a night
MIN_MINUTES_IN_BED = 120    # under two hours in bed is a nap misflagged as a main sleep
MIN_EFFICIENCY = 0.20       # below this the record is a device on a nightstand, not a night
MAX_EFFICIENCY = 1.00       # asleep cannot exceed in-bed; anything over 1 is a summing error

# Paired-window minimum, the same 7 the other wearable notebooks use, so the three stay
# comparable in the scorecard. Counted on ELIGIBLE nights rather than recorded ones, because a
# night the eligibility rule throws out cannot contribute to a baseline either.
PAIRED_MIN_DAYS = 7

coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                          AS cohort_patients,
        (SELECT COUNT(DISTINCT person_id) FROM sleep_nights)   AS patients_with_any_sleep,
        (SELECT COUNT(*) FROM sleep_nights)                    AS sleep_nights
"""

cov = cached_gbq(coverage_sql, "sleep_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

extract_sql = CTES + """
    SELECT person_id, rel_day, minute_in_bed, minute_asleep
    FROM sleep_nights
"""

# Dry run before executing: this is the one query in the notebook returning person-day rows
# rather than an aggregate, so it is also the one that can scan far more than expected.
cfg.dry_run_bytes(extract_sql)
nights = cached_gbq(extract_sql, "sleep_person_days")

# NOTHING BELOW PRINTS A ROW. `nights` and both grids are participant-level and stay in the
# kernel; every number printed or plotted from here on is a count or a quantile over the cohort.
nights["efficiency"] = nights["minute_asleep"] / nights["minute_in_bed"]
nights["duration_eligible"] = nights["minute_asleep"].between(
    MIN_MINUTES_ASLEEP, MAX_MINUTES_ASLEEP)
nights["efficiency_eligible"] = (
    nights["duration_eligible"]
    & nights["minute_in_bed"].ge(MIN_MINUTES_IN_BED)
    & nights["efficiency"].between(MIN_EFFICIENCY, MAX_EFFICIENCY)
)

# The unit each measure is reported in travels with the grid, so a figure label and a printed
# table cannot end up describing different quantities.
MEASURES = {
    "duration": {
        "grid": B.eligible_value_grid(nights, "minute_asleep", "duration_eligible",
                                      GRID_FIRST, REPORT_LAST),
        "label": "minutes asleep",
        "unit": "minutes",
        "fmt": "7.1f",
    },
    "efficiency": {
        "grid": B.eligible_value_grid(nights, "efficiency", "efficiency_eligible",
                                      GRID_FIRST, REPORT_LAST),
        "label": "sleep efficiency (minutes asleep / minutes in bed)",
        "unit": "efficiency points",
        "fmt": "7.4f",
    },
}

recorded = len(nights)
print(f"cohort patients                               : {cohort_patients:>10,}")
print(f"  ... with >=1 main sleep in the grid's span  : "
      f"{int(cov['patients_with_any_sleep']):>10,}"
      f"   ({int(cov['patients_with_any_sleep']) / cohort_patients:6.1%})")
print(f"recorded main-sleep nights in the grid's span : {recorded:>10,}")
print()

# Which bound did the dropping. A rule removing almost nothing is not protecting anything, and
# one removing a tenth of the data needs defending rather than inheriting.
for label, failed in [
    (f"asleep outside [{MIN_MINUTES_ASLEEP}, {MAX_MINUTES_ASLEEP}] min",
     ~nights["minute_asleep"].between(MIN_MINUTES_ASLEEP, MAX_MINUTES_ASLEEP)),
    (f"in bed < {MIN_MINUTES_IN_BED} min", ~nights["minute_in_bed"].ge(MIN_MINUTES_IN_BED)),
    (f"efficiency outside [{MIN_EFFICIENCY:.2f}, {MAX_EFFICIENCY:.2f}]",
     ~nights["efficiency"].between(MIN_EFFICIENCY, MAX_EFFICIENCY)),
]:
    n = int(failed.sum())
    print(f"  nights failing {label:<42s} {n:>9,}   ({n / recorded:6.2%})")

print()
print(f"{'measure':<14s}{'eligible nights':>17s}{'patients':>11s}"
      f"{'filled cells':>15s}{'paired ' + str(PAIRED_MIN_DAYS) + 'd':>13s}")
base_lo, base_hi = WINDOWS["baseline"]
post_lo, post_hi = WINDOWS["postop"]
for name, m in MEASURES.items():
    grid = m["grid"]
    # The paired count read off the grid rather than re-queried: a patient is paired when the
    # eligibility rule admits PAIRED_MIN_DAYS of their nights on each side of the surgery.
    baseline_days = grid.loc[base_lo:base_hi].notna().sum()
    postop_days = grid.loc[post_lo:post_hi].notna().sum()
    m["paired"] = int(((baseline_days >= PAIRED_MIN_DAYS)
                       & (postop_days >= PAIRED_MIN_DAYS)).sum())
    m["eligible_days"] = int(grid.notna().to_numpy().sum())
    print(f"{name:<14s}{m['eligible_days']:>17,}{grid.shape[1]:>11,}"
          f"{grid.notna().to_numpy().mean():>14.1%}{m['paired']:>13,}")

print()
for name, m in MEASURES.items():
    flag = "  << below threshold, do not report" if 0 < m["paired"] < SUPPRESS_THRESHOLD else ""
    print(f"  {name:<12s} paired population {m['paired']:>9,}"
          f"   ({m['paired'] / cohort_patients:6.1%} of cohort){flag}")

for name, m in MEASURES.items():
    m["coverage"] = reportable(m["grid"])
    srp_common.plot_rel_day_curve(
        cfg, m["coverage"].loc[m["coverage"]["rel_day"].between(GRID_FIRST, REPORT_LAST)],
        "n_patients",
        title=f"Patients contributing an eligible {name} night, by day relative to surgery",
        ylabel="patients with an eligible night", suppress=True, show_windows=True,
        save_as=f"{name}_grid_coverage")

print()
print(f"contributing patients on selected days, of {REPORT_LAST - REPORT_FIRST + 1} reported")
print(f"{'day':>6s}" + "".join(f"{name:>16s}" for name in MEASURES))
for day in [REPORT_FIRST, -14, -7, -1, 1, 7, 14, REPORT_LAST]:
    print(f"{day:>+6d}", end="")
    for m in MEASURES.values():
        row = m["coverage"].loc[m["coverage"]["rel_day"] == day]
        print(f"{'not reported':>16s}" if row.empty
              else f"{int(row['n_patients'].iloc[0]):>16,}", end="")
    print()

## Part 2 — What Each Baseline Calls Normal

A baseline is a claim about where a patient's normal sits, so the first thing to look at is the
claims themselves rather than the departures from them. Drawn as levels, all four are in the
measure's own unit and can share an axis, which is the only figure in this notebook where that is
true — and the observed cross-sectional curve is drawn beside them, dotted, as the thing they exist
to replace rather than as a result.

The four disagree in a way that is structural rather than noisy, and the shape of the disagreement
is the whole argument for choosing between them. The static baseline is one frozen number per
patient taken from before the surgery, so it is a flat line by construction and stays where the
patient was months earlier. The two trailing baselines follow the patient forward, which means that
after surgery they are partly built from post-operative nights and drift down toward whatever the
patient is now doing. **A trailing baseline therefore under-reports the very thing a recovery
metric is trying to measure, and a frozen one cannot adapt to a patient whose normal genuinely
changed.**

#### The Four Levels, Against the Night Actually Recorded

The static row of the movement table below is not expected to be exactly zero, and that is the
point. Each patient's static level is frozen, so any movement in it is purely the set of
contributing patients changing underneath a fixed set of values — **the composition effect itself,
measured**. It therefore bounds how much of the movement in the other three rows could be
composition rather than anything that happened to anybody.

In [ ]:
# Computed once and reused by Parts 3 and 4: the rolling medians over grids this size are the
# expensive step in the notebook, and nothing below them depends on anything but these.
for name, m in MEASURES.items():
    m["levels"] = B.baseline_levels(m["grid"], statistic=BASELINE_STATISTIC)
    m["deviations"] = B.deviations_from_each_baseline(m["grid"], statistic=BASELINE_STATISTIC)
    m["level_curves"] = {b: reportable(m["levels"][b]) for b in B.BASELINE_NAMES}
    m["departures"] = {b: reportable(m["deviations"][b]) for b in B.BASELINE_NAMES}

for name, m in MEASURES.items():
    # The observed nights themselves as the reference the four levels are read against. THIS IS
    # THE CROSS-SECTIONAL CURVE -- it moves when the set of contributing patients moves, which is
    # the problem the baselines exist to remove, so it is drawn as context and not as a result.
    series = {"observed night": m["coverage"],
              **{f"{b} baseline": m["level_curves"][b] for b in B.BASELINE_NAMES}}
    srp_common.plot_baseline_curves(
        cfg, series,
        title=f"Where each baseline puts a patient's normal {name}, "
              "against the night actually recorded",
        ylabel=m["label"], save_as=f"{name}_baseline_levels",
        span=(REPORT_FIRST, REPORT_LAST), colours=[INK_MUTED, *CATEGORICAL],
        note="median across contributing patients; levels only, not departures")

for name, m in MEASURES.items():
    print(f"{name}: median level across contributing patients, by window")
    print(f"  {'window':<22s}{'observed':>12s}"
          + "".join(f"{b:>14s}" for b in B.BASELINE_NAMES))
    for lo, hi, label in [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
                          (1, 7, "first post-op week"), (8, 30, "rest of the month")]:
        print(f"  {label:<22s}", end="")
        window = m["coverage"].loc[m["coverage"]["rel_day"].between(lo, hi), "p50"]
        print(f"{window.median():>{m['fmt']}}".rjust(12) if len(window)
              else f"{'--':>12s}", end="")
        for b in B.BASELINE_NAMES:
            frame = m["level_curves"][b]
            window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
            print(f"{window.median():>{m['fmt']}}".rjust(14) if len(window)
                  else f"{'--':>14s}", end="")
        print()
    print()

# How far each baseline has itself moved. A trailing baseline moving by an order more than the
# static one is moving for real reasons; one moving by about the same amount is not saying
# anything the composition effect does not already explain.
print("movement of the baseline itself, first post-op week vs the fortnight before")
for name, m in MEASURES.items():
    print(f"  {name} ({m['unit']})")
    for b in B.BASELINE_NAMES:
        frame = m["level_curves"][b]
        before = frame.loc[frame["rel_day"].between(-14, -1), "p50"].median()
        after = frame.loc[frame["rel_day"].between(1, 7), "p50"].median()
        if pd.isna(before) or pd.isna(after):
            print(f"    {b:<12s} not reportable in one of the two windows")
            continue
        print(f"    {b:<12s} {before:{m['fmt']}} -> {after:{m['fmt']}}"
              f"   ({after - before:+.4f})")

# The levels are only read in this cell. Dropping them here keeps two grids' worth of baselines
# out of the kernel for the rest of the notebook, which is the difference between this notebook
# fitting in a Workbench kernel and not.
for m in MEASURES.values():
    del m["levels"]

#### What Each Baseline Costs In Person-Days And In Patients

Every baseline above refuses to score a person-day it cannot serve: the static one drops patients
without enough pre-operative history, the trailing ones drop days whose window is too empty, and
the comparative one drops days where the long window has no spread to divide by. **A stricter
baseline is not a better one unless what it buys is worth the cohort it costs.** Read the patient
column first — a baseline keeping most person-days but few patients is being carried by a handful
of heavy device users.

In [ ]:
for name, m in MEASURES.items():
    cost = B.coverage_by_baseline(m["grid"], m["deviations"])
    m["cost"] = cost
    print(f"{name}: {m['eligible_days']:,} eligible person-days over "
          f"{m['grid'].shape[1]:,} patients")
    print(f"  {'baseline':<14s}{'person-days':>13s}{'of eligible':>13s}"
          f"{'patients':>11s}{'of eligible':>13s}")
    for _, row in cost.iterrows():
        # A baseline serving fewer patients than the threshold cannot be reported at all, let
        # alone used, so the count is checked here rather than only at plotting time.
        flag = ("   << below the small-cell threshold"
                if 0 < row["patients"] < SUPPRESS_THRESHOLD else "")
        print(f"  {row['baseline']:<14s}{int(row['person_days']):>13,}"
              f"{row['share_of_person_days']:>12.1%}"
              f"{int(row['patients']):>11,}{row['share_of_patients']:>12.1%}{flag}")
    print()

# The floors each refusal comes from, printed next to the losses they caused, so a cost can be
# read against the rule rather than against a number nobody can locate.
print("the floor doing the dropping, in each case")
print(f"  static      : {B.STATIC_MIN_ELIGIBLE_DAYS} eligible nights inside rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. {-B.STATIC_DAYS_PRIOR:+d}"
      f"   ({B.STATIC_MIN_ELIGIBLE_DAYS / B.STATIC_LOOKBACK_DAYS:.0%} of that window)")
print(f"  rolling     : {B.ROLLING_MIN_DAYS} eligible nights in the trailing "
      f"{B.ROLLING_WINDOW_DAYS} days")
print(f"  ewma        : {B.EWMA_MIN_EFFECTIVE_DAYS:.1f} effective days out of a capacity of "
      f"{B.ewma_full_effective_days():.1f}")
print(f"  comparative : {B.days_required(B.SHORT_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} "
      f"nights in the short window AND "
      f"{B.days_required(B.LONG_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} in the long one, "
      f"the long one also needing a non-zero IQR")

# The cost is not constant across the span and the tables above cannot show that. A trailing
# baseline is hungriest right after surgery, when the device is least likely to have been worn,
# so the days a recovery metric most needs are the days its baseline is most likely to refuse.
for name, m in MEASURES.items():
    srp_common.plot_baseline_coverage(
        cfg, m["grid"], m["deviations"], (REPORT_FIRST, REPORT_LAST),
        title=f"Patients each baseline can actually score for {name}, "
              "by day relative to surgery",
        save_as=f"{name}_baseline_coverage")

print()
print(f"thinnest reported day for each baseline, within rel_day {REPORT_FIRST:+d} .. "
      f"{REPORT_LAST:+d}")
for name, m in MEASURES.items():
    print(f"  {name}")
    for b in B.BASELINE_NAMES:
        scored = m["deviations"][b].notna().sum(axis=1).loc[REPORT_FIRST:REPORT_LAST]
        below = int((scored < SUPPRESS_THRESHOLD).sum())
        print(f"    {b:<12s} {int(scored.min()):>7,} patients on day "
              f"{int(scored.idxmin()):+d}"
              f"   ({below} of {len(scored)} days below the threshold)")

## Part 3 — Departure From Normal Around Surgery

This is what the baselines are for: how far each night sat from that patient's own normal, which is
the quantity a recovery metric would actually be built on. Differencing against a per-patient level
removes the composition effect a cross-sectional curve cannot, because a patient can only enter a
day's median by contributing a night **and** having a baseline of their own to be read against.

**Three of the four departures share a unit and the fourth does not.** Static, rolling and
exponential are a night's value minus a level, so they are in the measure's own unit; comparative
is a difference of two window centres divided by a spread, so it is in units of that patient's own
long-window IQR. They are drawn on separate axes for that reason, and a reader who puts them on one
axis will misread the fourth by whatever that patient's IQR happens to be.

The comparative curve also answers a slower question than the other three. Its numerator is the
last fourteen days rather than the night itself, so a short admission cannot appear on it at all,
and its response to surgery is smeared across the fortnight that follows rather than landing on the
days it happened. That is a property of the shape, not a defect to be tuned away.

#### The Departure Curves, and Whether Duration and Efficiency Say the Same Thing

Duration and efficiency are carried separately because they can disagree, and **a disagreement is
the finding rather than a nuisance**: minutes asleep falling while efficiency holds means the night
was truncated — a device coming off, or a ward waking someone early — whereas efficiency falling
while duration holds means the same time in bed bought less sleep. The day-by-day table through the
admission carries the contributing count beside every value, because a departure computed from
eight patients is a different kind of number from one computed from eight hundred.

In [ ]:
UNIT_BASELINES = ("static", "rolling", "ewma")   # the three in the measure's own unit

for name, m in MEASURES.items():
    srp_common.plot_baseline_curves(
        cfg, {f"{b} baseline": m["departures"][b] for b in UNIT_BASELINES},
        title=f"How far each night's {name} sat from that patient's own normal, "
              "by day relative to surgery",
        ylabel=f"{name} - that patient's baseline ({m['unit']})",
        save_as=f"{name}_departures", zero_line=True, span=(REPORT_FIRST, REPORT_LAST),
        note="median across contributing patients; zero is the night that patient usually has")

    srp_common.plot_baseline_curves(
        cfg, {f"{b} baseline": m["departures"][b] for b in UNIT_BASELINES},
        title=f"The same three {name} departures through the ten days either side of surgery",
        ylabel=f"{name} - that patient's baseline ({m['unit']})",
        save_as=f"{name}_departures_zoom", zero_line=True, span=ZOOM,
        note="median across contributing patients; zero is the night that patient usually has")

    # Drawn alone because its unit is not the others'. THE Y AXIS HERE IS IQRs OF THAT PATIENT'S
    # OWN LONG WINDOW, not the measure's own unit.
    srp_common.plot_baseline_curves(
        cfg, {"comparative index": m["departures"]["comparative"]},
        title=f"The recent fortnight of {name} against the established normal, "
              "in that patient's own IQRs",
        ylabel="(short centre - long centre) / long IQR",
        save_as=f"{name}_comparative_index", zero_line=True,
        span=(REPORT_FIRST, REPORT_LAST),
        note="median across contributing patients; unitless, not comparable to the other figures")

WINDOW_ROWS = [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
               (1, 3, "first three days"), (4, 7, "rest of week one"),
               (8, 14, "week two"), (15, 30, "weeks three and four")]

for name, m in MEASURES.items():
    print(f"{name}: median departure from baseline, by window")
    print(f"  {'window':<24s}" + "".join(f"{b:>14s}" for b in B.BASELINE_NAMES))
    for lo, hi, label in WINDOW_ROWS:
        print(f"  {label:<24s}", end="")
        for b in B.BASELINE_NAMES:
            frame = m["departures"][b]
            window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
            print(f"{window.median():>+14.4f}" if len(window) else f"{'--':>14s}", end="")
        print()
    print(f"  (the first three columns are {m['unit']}; comparative is long-window IQRs)")
    print()

# Day by day through the admission, with the contributing count beside each value. The two
# measures are printed side by side because the question this section asks -- whether duration
# and efficiency move together -- cannot be read off two separate tables.
print("the ten days either side of surgery, static baseline, both measures")
print(f"{'day':>5s}" + "".join(f"{name:>14s}{'n':>9s}" for name in MEASURES))
for day in range(ZOOM[0], ZOOM[1] + 1):
    print(f"{day:>+5d}", end="")
    for m in MEASURES.values():
        row = m["departures"]["static"].loc[m["departures"]["static"]["rel_day"] == day]
        if row.empty:
            print(f"{'--':>14s}{'--':>9s}", end="")
        else:
            print(f"{row['p50'].iloc[0]:>+14.4f}{int(row['n_patients'].iloc[0]):>9,}", end="")
    print()

## Part 4 — Whether the Choice of Baseline Changes the Answer

Two baselines that rank person-days the same way are interchangeable for anything downstream that
only cares about which nights were worse than usual, whatever their levels look like side by side.
Spearman is the right coefficient here because the comparative index is on a different scale
entirely, so only the **ordering** of person-days is comparable across a pair. Each pair is
computed on the person-days both baselines score, not on either one's full set, so the counts
differ from the coverage table in Part 2.

Rank correlation says nothing about **where zero is**, and zero is the whole claim — "this night
was worse than that patient's normal". Two baselines can rank a month of nights identically and
still put a third of them on opposite sides of normal, because one sits lower than the other
throughout, and a metric counting bad nights would then count different nights. Both questions are
asked, and both are restricted to the post-operative month as well as the whole grid, because a
pair can agree across a year of ordinary nights and disagree exactly where the surgery is.

#### Rank Agreement, Side of Normal, and the Share Called Abnormal

The two thresholds are conventions and are named in the cell rather than left to the reader's
judgement. **A baseline calling most of the post-operative month abnormal is not detecting
recovery, it is mis-centred** — which is why the last table is the share of scored post-operative
person-days each baseline puts below normal, and why a value near a half is the reassuring one
rather than a large one.

In [ ]:
# Above INTERCHANGEABLE the pair ranks days the same and either will do; below DISTINCT the two
# are answering different questions and the choice between them IS the analysis.
INTERCHANGEABLE = 0.95
DISTINCT = 0.70
POSTOP = WINDOWS["postop"]

for name, m in MEASURES.items():
    postop_only = {b: dev.loc[POSTOP[0]:POSTOP[1]] for b, dev in m["deviations"].items()}
    m["postop_only"] = postop_only
    whole = B.baseline_agreement(m["deviations"])
    post = B.baseline_agreement(postop_only)

    print(f"{name}: pairwise Spearman on the person-days both baselines score")
    print(f"  {'pair':<28s}{'whole grid':>13s}{'n':>14s}"
          f"{'postop only':>14s}{'n':>14s}")
    for (_, w), (_, p) in zip(whole.iterrows(), post.iterrows()):
        print(f"  {w['pair']:<28s}{w['spearman']:>13.3f}{int(w['person_days']):>14,}"
              f"{p['spearman']:>14.3f}{int(p['person_days']):>14,}")

    print(f"  reading, on rel_day {POSTOP[0]:+d} .. {POSTOP[1]:+d}")
    for _, row in post.iterrows():
        rho = row["spearman"]
        if pd.isna(rho):
            verdict = "not enough shared person-days to say"
        elif rho >= INTERCHANGEABLE:
            verdict = "interchangeable for anything that only ranks days"
        elif rho >= DISTINCT:
            verdict = "related but not substitutable"
        else:
            verdict = "different questions; the choice between them IS the analysis"
        print(f"    {row['pair']:<28s} {rho:>6.3f}   {verdict}")
    print()

print(f"share of shared person-days a pair puts on the SAME side of normal, rel_day "
      f"{POSTOP[0]:+d} .. {POSTOP[1]:+d}")
for name, m in MEASURES.items():
    print(f"  {name}")
    print(f"    {'pair':<28s}{'person-days':>14s}{'same side':>12s}{'both below':>12s}")
    for i, left in enumerate(B.BASELINE_NAMES):
        for right in B.BASELINE_NAMES[i + 1:]:
            a, b = B.flatten_pair(m["postop_only"][left], m["postop_only"][right])
            if len(a) < SUPPRESS_THRESHOLD:
                print(f"    {left + ' vs ' + right:<28s}{len(a):>14,}"
                      f"{'--':>12s}{'--':>12s}   too few shared person-days")
                continue
            print(f"    {left + ' vs ' + right:<28s}{int(len(a)):>14,}"
                  f"{float(((a < 0) == (b < 0)).mean()):>12.1%}"
                  f"{float(((a < 0) & (b < 0)).mean()):>12.1%}")

# The single number a downstream metric would care about most, and the one that catches a
# mis-centred baseline: how much of the post-operative month each one calls abnormal at all.
print()
print("share of scored post-operative person-days each baseline calls below normal")
for name, m in MEASURES.items():
    print(f"  {name}")
    for b in B.BASELINE_NAMES:
        values = B.flatten(m["postop_only"][b])
        if len(values) < SUPPRESS_THRESHOLD:
            print(f"    {b:<12s} too few scored person-days to report")
            continue
        print(f"    {b:<12s} {float((values < 0).mean()):>6.1%}"
              f"   of {len(values):,} person-days")

## Part 5 — Verdict

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. **Nothing written here is a
participant-level value**: the row is counts, fractions and hand-entered judgements.

#### The Scorecard Row

Face validity is the weak point for this metric rather than coverage, and that judgement is
recorded in the row's notes so a later reader can disagree with it specifically. **The analysable
population is now the paired count on eligible nights**, which is stricter than the old any-record
paired count, so this row is not comparable to one written before the baselines were integrated —
the per-baseline scored counts beside it are what say how much of that population a given baseline
would actually serve.

In [ ]:
# Reported on efficiency, which is the measure the baselines were applied to first and the one a
# sleep-based recovery metric would most likely be built on; duration's numbers are in the row
# beside it rather than in a second scorecard, because they describe one candidate metric.
primary = MEASURES["efficiency"]
scored = {b: int(primary["cost"].set_index("baseline").loc[b, "patients"])
          for b in B.BASELINE_NAMES}

srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="sleep_daily_summary",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    outcome_definition=(
        "nightly sleep efficiency (minutes asleep / minutes in bed over main sleeps) read "
        "against that patient's own baseline; four baselines compared, none yet settled"
    ),
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_sleep"]),
    patients_analysable=primary["paired"],
    coverage_share=primary["paired"] / cohort_patients,
    horizon_days=int(WINDOWS["postop"][1]),
    duration_patients_analysable=MEASURES["duration"]["paired"],
    baseline_choice="unsettled",
    patients_scored_by_baseline="; ".join(f"{b}={scored[b]:,}" for b in B.BASELINE_NAMES),
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Read within patient against baselines.py, not cross-sectionally. Face validity is "
           "the weak point, not coverage: ward noise, analgesia and a device left off overnight "
           "all move sleep without being recovery. Baseline parameters are inherited from a "
           "cancer-diagnosis sweep and have not been re-swept against this cohort."),
)